In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))
print("✅ Hazır!")

Mounted at /content/drive
TF version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ Hazır!


In [ ]:
base = '/content/drive/MyDrive/KemikAI'

# Görüntü girdisi
img_input = layers.Input(shape=(224, 224, 3), name='image_input')

# EfficientNetV2S backbone
backbone = EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_tensor=img_input
)
backbone.trainable = False  # Önce dondur

x = backbone.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
img_features = layers.Dense(128, activation='relu', name='img_features')(x)

# Klinik veri girdisi (9 özellik)
clinical_input = layers.Input(shape=(9,), name='clinical_input')
y = layers.Dense(64, activation='relu')(clinical_input)
y = layers.Dropout(0.2)(y)
clinical_features = layers.Dense(32, activation='relu', name='clinical_features')(y)

# Fusion katmanı
combined = layers.Concatenate()([img_features, clinical_features])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
output = layers.Dense(1, name='bone_age_output')(z)

# Model
model = Model(
    inputs=[img_input, clinical_input],
    outputs=output,
    name='KemikAI'
)

model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='mae',
    metrics=['mae']
)

print("✅ Model kuruldu!")
print(f"Toplam parametre: {model.count_params():,}")
model.summary()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
✅ Model kuruldu!
Toplam parametre: 20,715,649


Model: "KemikAI"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 112, 112,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 112, 112,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 112, 112,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 112, 112,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 56, 56,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 56, 56,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 56, 56,    │          0 │ block2a_expand_b

 Total params: 20,715,649 (79.02 MB)

 Trainable params: 384,289 (1.47 MB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [ ]:
class KemikAIGenerator(Sequence):
    def __init__(self, df, img_path, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.img_path = img_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(df))

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        imgs, clinical, labels = [], [], []

        for i in batch_idx:
            row = self.df.iloc[i]
            img_file = f'{self.img_path}/{int(row.ID)}.png'

            img = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((224, 224), dtype=np.float32)
            else:
                img = cv2.resize(img, (224, 224))

            img = img.astype(np.float32) / 255.0
            img = np.stack([img]*3, axis=-1)
            imgs.append(img)
            clinical.append(row[clinical_cols].values.astype(np.float32))
            labels.append(row['Boneage'])

        return {'image_input': np.array(imgs),
                'clinical_input': np.array(clinical)}, np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

train_gen = KemikAIGenerator(train_df, train_img_path, batch_size=32)
val_gen = KemikAIGenerator(val_df, train_img_path, batch_size=32, shuffle=False)

print("✅ Generator hazır!")

✅ Generator hazır!


In [ ]:
# CSV'deki ID örnekleri
print("CSV ID örnekleri:", df['ID'].values[:5])

# Klasördeki dosya örnekleri
print("Klasör dosyaları:", os.listdir(train_img_path)[:5])

# ID var mı klasörde?
test_id = df['ID'].values[0]
test_path = f'{train_img_path}/{test_id}.png'
print(f"\nTest path: {test_path}")
print(f"Dosya var mı: {os.path.exists(test_path)}")

CSV ID örnekleri: [1377 1378 1379 1380 1381]
Klasör dosyaları: ['8086.png', '8087.png', '8088.png', '8089.png', '8091.png']

Test path: /content/drive/MyDrive/KemikAI/data/raw/RSNA_train/images/1377.png
Dosya var mı: True


In [ ]:
import shutil
print("Kopyalanıyor... (5-10 dk bekle)")
shutil.copytree(
    '/content/drive/MyDrive/KemikAI/data/raw/RSNA_train/images',
    '/content/train_images'
)
print("✅ Kopyalandı!")

Kopyalanıyor... (5-10 dk bekle)
✅ Kopyalandı!


In [ ]:
# Yeni path
train_img_path = '/content/train_images'

# Generator'ları yenile
train_gen = KemikAIGenerator(train_df, train_img_path, batch_size=32)
val_gen = KemikAIGenerator(val_df, train_img_path, batch_size=32, shuffle=False)

print("✅ Hazır, eğitim başlıyor!")

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

✅ Hazır, eğitim başlıyor!


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 864ms/step - loss: 48.0719 - mae: 48.0719

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()



Epoch 1: val_mae improved from None to 9.55816, saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model.keras
315/315 ━━━━━━━━━━━━━━━━━━━━ 356s 1s/step - loss: 28.0198 - mae: 28.0198 - val_loss: 9.5582 - val_mae: 9.5582 - learning_rate: 0.0010
Epoch 2/20
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 862ms/step - loss: 14.9397 - mae: 14.9397
Epoch 2: val_mae did not improve from 9.55816
315/315 ━━━━━━━━━━━━━━━━━━━━ 338s 1s/step - loss: 14.7324 - mae: 14.7324 - val_loss: 9.9852 - val_mae: 9.9852 - learning_rate: 0.0010
Epoch 3/20
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 855ms/step - loss: 14.4053 - mae: 14.4053
Epoch 3: val_mae did not improve from 9.55816
315/315 ━━━━━━━━━━━━━━━━━━━━ 338s 1s/step - loss: 14.4450 - mae: 14.4450 - val_loss: 10.1864 - val_mae: 10.1864 - learning_rate: 0.0010
Epoch 4/20
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 861ms/step - loss: 13.9526 - mae: 13.9526
Epoch 4: val_ma

KeyboardInterrupt: 

In [ ]:
# En iyi modeli yükle
from tensorflow import keras
model = keras.models.load_model(f'{base}/models/checkpoints/best_model.keras')

# Backbone'u aç
model.trainable = True

# Sadece son 30 katmanı eğit
for layer in model.layers[:-30]:
    layer.trainable = False

# Daha düşük learning rate ile derle
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='mae',
    metrics=['mae']
)

trainable = sum([layer.trainable for layer in model.layers])
print(f"✅ Eğitilebilir katman: {trainable}")
print("Fine-tune başlıyor...")

history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

✅ Eğitilebilir katman: 12
Fine-tune başlıyor...
Epoch 1/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 869ms/step - loss: 13.6417 - mae: 13.6417
Epoch 1: val_mae improved from 8.44850 to 8.32402, saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model.keras
315/315 ━━━━━━━━━━━━━━━━━━━━ 416s 1s/step - loss: 13.4907 - mae: 13.4907 - val_loss: 8.3240 - val_mae: 8.3240 - learning_rate: 1.0000e-05
Epoch 2/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 871ms/step - loss: 13.5462 - mae: 13.5462
Epoch 2: val_mae did not improve from 8.32402
315/315 ━━━━━━━━━━━━━━━━━━━━ 344s 1s/step - loss: 13.5392 - mae: 13.5392 - val_loss: 8.3334 - val_mae: 8.3334 - learning_rate: 1.0000e-05
Epoch 3/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 872ms/step - loss: 13.3594 - mae: 13.3594
Epoch 3: val_mae improved from 8.32402 to 8.32234, saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model.keras

Epoc

KeyboardInterrupt: 

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

base = '/content/drive/MyDrive/KemikAI'

# Modeli sıfırdan kur — backbone tamamen açık
img_input = layers.Input(shape=(224, 224, 3), name='image_input')

backbone = EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_tensor=img_input
)
backbone.trainable = True  # Tamamen açık

x = backbone.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.4)(x)
img_features = layers.Dense(256, activation='relu')(x)

clinical_input = layers.Input(shape=(9,), name='clinical_input')
y = layers.Dense(128, activation='relu')(clinical_input)
y = layers.BatchNormalization()(y)
y = layers.Dropout(0.3)(y)
clinical_features = layers.Dense(64, activation='relu')(y)

combined = layers.Concatenate()([img_features, clinical_features])
z = layers.Dense(256, activation='relu')(combined)
z = layers.Dropout(0.3)(z)
z = layers.Dense(64, activation='relu')(z)
output = layers.Dense(1, name='bone_age_output')(z)

model = Model(inputs=[img_input, clinical_input], outputs=output, name='KemikAI_v2')

# İki aşamalı learning rate
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='mae',
    metrics=['mae']
)

callbacks = [
    ModelCheckpoint(
        f'{base}/models/checkpoints/best_model_v2.keras',
        monitor='val_mae', save_best_only=True, verbose=1
    ),
    EarlyStopping(monitor='val_mae', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_mae', factor=0.3, patience=3, min_lr=1e-7, verbose=1)
]

print(f"Toplam parametre: {model.count_params():,}")
print("Eğitim başlıyor...")

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

# Modeli kaydet
model.save(f'{base}/models/final/kemikai_final.keras')
print("✅ Model kaydedildi!")

Toplam parametre: 21,232,417
Eğitim başlıyor...
Epoch 1/30
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 976ms/step - loss: 67.5686 - mae: 67.5686
Epoch 1: val_mae improved from None to 42.15074, saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model_v2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model_v2.keras
315/315 ━━━━━━━━━━━━━━━━━━━━ 596s 1s/step - loss: 40.8191 - mae: 40.8191 - val_loss: 42.1507 - val_mae: 42.1507 - learning_rate: 1.0000e-04
Epoch 2/30
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - loss: 19.0358 - mae: 19.0358
Epoch 2: val_mae improved from 42.15074 to 9.23771, saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model_v2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/KemikAI/models/checkpoints/best_model_v2.keras
315/315 ━━━━━━━━━━━━━━━━━━━━ 375s 1s/step - loss: 17.4695 - mae: 17.4695 - val_loss: 9.2377 - val_mae: 9.2377 - learning_rate: 1.0000e-04
Epoch 3/30
315/315 ━━

KeyboardInterrupt: 

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
from tensorflow.keras.utils import Sequence
import cv2

base = '/content/drive/MyDrive/KemikAI'
train_img_path = '/content/train_images'

# Sadece görüntü + cinsiyet kullanan generator
class SimpleGenerator(Sequence):
    def __init__(self, df, img_path, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.img_path = img_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(df))

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        imgs, labels = [], []

        for i in batch_idx:
            row = self.df.iloc[i]
            img = cv2.imread(f'{self.img_path}/{int(row.ID)}.png', cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.zeros((224, 224), dtype=np.float32)
            img = cv2.resize(img, (224, 224)).astype(np.float32) / 255.0
            # Cinsiyeti görüntüye encode et
            if row.gender == 1:
                img = np.stack([img, img, np.ones_like(img)], axis=-1)
            else:
                img = np.stack([img, np.ones_like(img), img], axis=-1)
            imgs.append(img)
            labels.append(row['Boneage'])

        return np.array(imgs), np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

train_gen2 = SimpleGenerator(train_df, train_img_path, batch_size=16)
val_gen2 = SimpleGenerator(val_df, train_img_path, batch_size=16, shuffle=False)

# Model
base_model = EfficientNetV2S(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(1024, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.2)(x)
output = layers.Dense(1)(x)

model2 = Model(inputs=base_model.input, outputs=output, name='KemikAI_v3')
model2.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='mae',
    metrics=['mae']
)

callbacks2 = [
    ModelCheckpoint(
        f'{base}/models/checkpoints/best_v3.keras',
        monitor='val_mae', save_best_only=True, verbose=1
    ),
    EarlyStopping(monitor='val_mae', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

print("Eğitim başlıyor...")
history3 = model2.fit(
    train_gen2,
    validation_data=val_gen2,
    epochs=20,
    callbacks=callbacks2,
    verbose=1
)

model2.save(f'{base}/models/final/kemikai_v3.keras')
print("✅ Bitti!")

Eğitim başlıyor...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
528/630 ━━━━━━━━━━━━━━━━━━━━ 45s 448ms/step - loss: 61.0358 - mae: 61.0358

KeyboardInterrupt: 

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import pandas as pd

# ==========================================
# 1. KONFİGÜRASYON VE PARAMETRELER
# ==========================================
IMAGE_DIR = '/content/train_images'  # Görüntülerin olduğu klasör
MODEL_SAVE_DIR = '/content/drive/MyDrive/KemikAI/models'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Klasör yoksa oluştur
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# NOT: Bu kodun çalışması için 'train_df' ve 'val_df' adında
# pandas dataframe'lerinin hafızada olması gerekmektedir.
# Eğer sadece 'train_df' varsa ve ayırmadıysanız, aşağıdaki kodu açın:
# from sklearn.model_selection import train_test_split
# train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

# ==========================================
# 2. VERİ YÜKLEME (DATA PIPELINE)
# ==========================================
def process_path(file_path, gender, label):
    # Görüntüyü oku ve decode et
    img = tf.io.read_file(file_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    # EfficientNetV2 için ekstra normalization gerekmez (kendi içinde yapar)
    return (img, gender), label

def create_tf_dataset(df, is_training=True):
    # Dosya yollarını oluştur
    # ID kolonunda .png uzantısı yoksa ekle, varsa ekleme
    df = df.copy()
    df['file_path'] = df['ID'].apply(
        lambda x: os.path.join(IMAGE_DIR, f"{x}.png" if not str(x).endswith('.png') else str(x))
    )

    # Hata almamak için: Sadece klasörde gerçekten var olan dosyaları filtrele
    exists_mask = df['file_path'].apply(os.path.exists)
    missing_count = (~exists_mask).sum()
    if missing_count > 0:
        print(f"UYARI: {missing_count} adet dosya '{IMAGE_DIR}' klasöründe bulunamadı. Eğitimden çıkarılıyor...")
        df = df[exists_mask]

    file_paths = df['file_path'].values
    genders = df['gender'].values.astype(np.float32)
    labels = df['Boneage'].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((file_paths, genders, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(file_paths))

    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("Veri setleri oluşturuluyor...")
train_dataset = create_tf_dataset(train_df, is_training=True)
val_dataset = create_tf_dataset(val_df, is_training=False)

# ==========================================
# 3. VERİ ARTIRMA (DATA AUGMENTATION)
# ==========================================
# Kemik yaşında yatay çevirme (flip) kullanılmamalıdır.
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(factor=0.05, fill_mode='nearest'),
    layers.RandomZoom(height_factor=0.1, width_factor=0.1, fill_mode='nearest'),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05, fill_mode='nearest'),
    layers.RandomBrightness(factor=0.1),
    layers.RandomContrast(factor=0.1)
], name="data_augmentation")

# ==========================================
# 4. MODEL MİMARİSİ (DUAL INPUT)
# ==========================================
def build_model():
    # 1. Girdi: Görüntü
    image_input = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name='image_input')
    x = data_augmentation(image_input)

    # Base Model (EfficientNetV2S)
    base_model = EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_tensor=x,
        include_preprocessing=True # Keras 3/2.10+ için otomatik preprocessing
    )
    # İlk aşamada base_model ağırlıklarını dondur
    base_model.trainable = False

    # Görüntü Özellikleri
    image_features = base_model.output
    image_features = layers.GlobalAveragePooling2D()(image_features)

    # 2. Girdi: Cinsiyet (0 veya 1)
    gender_input = layers.Input(shape=(1,), name='gender_input')
    gender_features = layers.Dense(16, activation='swish')(gender_input)

    # Özellikleri Birleştir
    concat = layers.Concatenate()([image_features, gender_features])

    # Sınıflandırıcı Başlık (Head)
    y = layers.Dense(512, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(concat)
    y = layers.Dropout(0.3)(y)
    y = layers.Dense(256, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(y)
    y = layers.Dropout(0.2)(y)
    output = layers.Dense(1, activation='linear', name='boneage_output')(y)

    model = models.Model(inputs=[image_input, gender_input], outputs=output)
    return model, base_model

model, base_model = build_model()
model.summary()

# ==========================================
# 5. EĞİTİM - AŞAMA 1 (Sadece Head Eğitimi)
# ==========================================
print("\n--- AŞAMA 1: Yeni Katmanların Eğitimi (Warm-up) ---")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.Huber(), # Aykırı değerlere MSE'den daha dayanıklıdır
    metrics=['mae']
)

history_phase1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5
)

# ==========================================
# 6. EĞİTİM - AŞAMA 2 (Fine-tuning)
# ==========================================
print("\n--- AŞAMA 2: Tüm Modelin İnce Ayarı (Fine-tuning) ---")
# Base modelin tüm katmanlarını çöz (unfreeze)
base_model.trainable = True

# Callbacks
model_checkpoint = ModelCheckpoint(
    filepath=os.path.join(MODEL_SAVE_DIR, 'kemikai_effv2s_best.keras'),
    monitor='val_mae',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_mae',
    patience=8,
    restore_best_weights=True,
    mode='min',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_mae',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    mode='min',
    verbose=1
)

# Çok daha düşük bir öğrenme oranıyla derle
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.Huber(),
    metrics=['mae']
)

history_phase2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25,
    callbacks=[model_checkpoint, early_stopping, reduce_lr]
)

print("\nEğitim tamamlandı! En iyi model Drive'a kaydedildi.")


Veri setleri oluşturuluyor...
UYARI: 585 adet dosya '/content/train_images' klasöründe bulunamadı. Eğitimden çıkarılıyor...
UYARI: 146 adet dosya '/content/train_images' klasöründe bulunamadı. Eğitimden çıkarılıyor...


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 224, 224,  │          0 │ data_augmentatio… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        648 │ rescaling_4[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 112, 112,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 112, 112,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 112, 112,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 112, 112,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 112, 112,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 112, 112,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 112, 112,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 56, 56,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 56, 56,    │        384 │ block2a_expand_c

 Total params: 21,127,041 (80.59 MB)

 Trainable params: 795,681 (3.04 MB)

 Non-trainable params: 20,331,360 (77.56 MB)


--- AŞAMA 1: Yeni Katmanların Eğitimi (Warm-up) ---
Epoch 1/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 352s 1s/step - loss: 30.3265 - mae: 30.6891 - val_loss: 20.7233 - val_mae: 21.0671
Epoch 2/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 313s 1s/step - loss: 21.9041 - mae: 22.2333 - val_loss: 17.5480 - val_mae: 17.8609
Epoch 3/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - loss: 20.3992 - mae: 20.7028 - val_loss: 16.9714 - val_mae: 17.2614
Epoch 4/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 334s 1s/step - loss: 19.6661 - mae: 19.9487 - val_loss: 22.5543 - val_mae: 22.8309
Epoch 5/5
297/297 ━━━━━━━━━━━━━━━━━━━━ 336s 1s/step - loss: 19.7451 - mae: 20.0087 - val_loss: 15.6161 - val_mae: 15.8697

--- AŞAMA 2: Tüm Modelin İnce Ayarı (Fine-tuning) ---
Epoch 1/25
297/297 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 49.0910 - mae: 49.3462
Epoch 1: val_mae improved from None to 12.33607, saving model to /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/KemikAI/model

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import zipfile

zip_path = '/content/drive/MyDrive/KemikAI/data/raw/rsna-bone-age.zip'
extract_path = '/content/train_images'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Bitti!")

✅ Bitti!


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/KemikAI/data/raw/RSNA_Annotations/RSNA_Annotations/BONEAGE/boneage_train.csv')

# gender kolonu ekle
df['gender'] = df['Male'].apply(lambda x: 1 if x else 0)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Eğitim verisi: {len(train_df)}")
print(f"Doğrulama verisi: {len(val_df)}")
print(df.head())


Eğitim verisi: 10088
Doğrulama verisi: 2523
     ID   Male  Boneage  gender
0  1377  False      180       0
1  1378  False       12       0
2  1379  False       94       0
3  1380   True      120       1
4  1381  False       82       0


In [10]:
import os
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import pandas as pd

# ==========================================
# 1. KONFİGÜRASYON VE PARAMETRELER
# ==========================================
IMAGE_DIR = '/content/train_images'
MODEL_SAVE_DIR = '/content/drive/MyDrive/KemikAI/models'
MODEL_PATH = os.path.join(MODEL_SAVE_DIR, 'kemikai_effv2s_best.keras')
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ==========================================
# 2. VERİ YÜKLEME (AYNI KOD)
# ==========================================
def process_path(file_path, gender, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return (img, gender), label

def get_existing_path(x):
    filename = f"{x}.png" if not str(x).endswith('.png') else str(x)
    path1 = os.path.join(IMAGE_DIR, 'RSNA_train', 'images', filename)
    path2 = os.path.join(IMAGE_DIR, 'RSNA_val', 'images', filename)
    path3 = os.path.join(IMAGE_DIR, 'RSNA_train', filename)
    path4 = os.path.join(IMAGE_DIR, 'RSNA_val', filename)
    path5 = os.path.join(IMAGE_DIR, filename)

    if os.path.exists(path1): return path1
    if os.path.exists(path2): return path2
    if os.path.exists(path3): return path3
    if os.path.exists(path4): return path4
    if os.path.exists(path5): return path5
    return path1

def create_tf_dataset(df, is_training=True):
    df = df.copy()
    df['file_path'] = df['ID'].apply(get_existing_path)

    exists_mask = df['file_path'].apply(os.path.exists)
    missing_count = (~exists_mask).sum()
    if missing_count > 0:
        print(f"UYARI: {missing_count} adet dosya bulunamadı. (RSNA_train veya RSNA_val içinde yok)")
        df = df[exists_mask]

    file_paths = df['file_path'].values
    genders = df['gender'].values.astype(np.float32)
    labels = df['Boneage'].values.astype(np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((file_paths, genders, labels))

    if is_training:
        dataset = dataset.shuffle(buffer_size=len(file_paths))

    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("Veri setleri oluşturuluyor...")
train_dataset = create_tf_dataset(train_df, is_training=True)
val_dataset = create_tf_dataset(val_df, is_training=False)

# ==========================================
# 3. MODELİ DRIVE'DAN YÜKLEME VE DEVAM ETME
# ==========================================
print(f"\nModel Drive'dan yükleniyor: {MODEL_PATH}")
# Modeli yükle (Kayıtlı durumuyla)
model = models.load_model(MODEL_PATH)

# BatchNormalization katmanlarını yine dondurmaya devam edelim (güvenlik için)
for layer in model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Modelin optimizer'ı ve loss'u zaten yüklendiği için tekrar compile etmeye gerek yok,
# ancak lr'yi hafifçe düşürüp yeniden derlemek daha güvenli olabilir.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), # Öğrenme oranını biraz daha düşürdük
    loss=tf.keras.losses.Huber(),
    metrics=['mae']
)

# Callbacks
model_checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor='val_mae',
    save_best_only=True,
    mode='min',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_mae',
    patience=8,
    restore_best_weights=True,
    mode='min',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_mae',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    mode='min',
    verbose=1
)

print("\n--- EĞİTİME KALDIĞI YERDEN (EPOCH 2) DEVAM EDİLİYOR ---")
history_resume = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=24, # 1 epoch zaten bittiği için 24 epoch daha yapıyoruz
    callbacks=[model_checkpoint, early_stopping, reduce_lr]
)
print("\nEğitim başarıyla tamamlandı!")


Veri setleri oluşturuluyor...

Model Drive'dan yükleniyor: /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras

--- EĞİTİME KALDIĞI YERDEN (EPOCH 2) DEVAM EDİLİYOR ---
Epoch 1/24
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 935ms/step - loss: 13.3891 - mae: 13.6405
Epoch 1: val_mae improved from None to 10.31622, saving model to /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras
316/316 ━━━━━━━━━━━━━━━━━━━━ 453s 1s/step - loss: 12.7533 - mae: 13.0043 - val_loss: 10.0694 - val_mae: 10.3162 - learning_rate: 5.0000e-05
Epoch 2/24
316/316 ━━━━━━━━━━━━━━━━━━━━ 0s 971ms/step - loss: 11.6971 - mae: 11.9454
Epoch 2: val_mae improved from 10.31622 to 9.34011, saving model to /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/KemikAI/models/kemikai_effv2s_best.keras
316/316 ━━━━━━━━━━━━━━━━━━━━ 421s 1s/step - loss: 1